# Data Cleaning and Preprocessing

## Objective

This notebook turns the engineered match-level dataset from Notebook 01 into
three numeric datasets that are ready for modelling.

We will:

1. keep only predictors that are valid before a match begins;
2. check data quality without changing the engineered source file;
3. split matches chronologically into training, validation, and test seasons;
4. learn categorical encoding from the training data only; and
5. save the three processed splits for Notebook 04.

**Input:** `data/processed/model_data_engineered.csv`

**Outputs:**

- `data/processed/train_processed.csv`
- `data/processed/validation_processed.csv`
- `data/processed/test_processed.csv`

The 2015/2016 test season is prepared and saved, but it is not used to make
preprocessing, feature-selection, or model-comparison decisions.


## 1. Imports and Project Paths

We first import the small set of tools needed for tabular preprocessing.
The path search makes the notebook work when it is launched either from the
repository root or from the `notebooks` folder.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder


# Find the repository by looking for this notebook's engineered input file.
current_directory = Path.cwd().resolve()
project_root = None

for candidate in (current_directory, *current_directory.parents):
    candidate_input = candidate / "data" / "processed" / "model_data_engineered.csv"
    if candidate_input.exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find data/processed/model_data_engineered.csv. "
        "Run this notebook from inside the project repository."
    )

processed_data_directory = project_root / "data" / "processed"
engineered_data_path = processed_data_directory / "model_data_engineered.csv"

print("Project root:", project_root)
print("Input file:", engineered_data_path.relative_to(project_root))


Project root resolved successfully.
Input file: data/processed/model_data_engineered.csv


## 2. Load the Engineered Dataset

One row represents one football match. The columns combine match metadata,
the result label, and information that was available before kickoff, such as
historical team attributes, player summaries, and recent form.

Loading the file into a new DataFrame does not modify the source CSV.


In [2]:
match_data = pd.read_csv(engineered_data_path)

# Dates are metadata, but converting them now lets us verify time ordering.
match_data["date"] = pd.to_datetime(match_data["date"], errors="raise")

dataset_overview = pd.DataFrame(
    {
        "Measure": ["Rows", "Columns", "Earliest match", "Latest match"],
        "Value": [
            len(match_data),
            match_data.shape[1],
            match_data["date"].min().date(),
            match_data["date"].max().date(),
        ],
    }
)

display(dataset_overview)

assert match_data.shape == (17609, 156), "The engineered dataset shape has changed."


,Measure,Value
0,Rows,17609
1,Columns,156
2,Earliest match,2010-07-17
3,Latest match,2016-05-25


### Finding

The engineered dataset contains **17,609 matches and 156 columns**, covering
July 2010 to May 2016. Not all 156 columns are legitimate model inputs.
Several are identifiers, descriptive metadata, post-match results, or raw
fields retained only to document how the engineered row was built.


## 3. Define the 75 Valid Candidate Predictors

The following removal is **conceptual filtering**, not statistical feature
selection. A column is excluded because of what it means, not because of its
correlation or model score.

- **Identifiers** label records or teams; their numeric values do not measure
  football ability.
- **Metadata** describes the match but is not an intended predictor here.
- **Post-match leakage variables** contain the goals that determine the
  target and are unknown before kickoff.
- **Raw player IDs** identify people rather than summarise their ability.
- **Formation coordinates** are raw lineup locations retained from the source.
- **Attribute dates** were needed for historical joins in Notebook 01, but
  are not model features.

The target is separated because it is the value the models will predict.


In [3]:
identifier_columns = [
    "id",
    "match_api_id",
    "country_id",
    "league_id",
    "home_team_api_id",
    "away_team_api_id",
]

metadata_columns = [
    "date",
    "country_name",
    "home_team_name",
    "away_team_name",
]

post_match_leakage_columns = [
    "home_team_goal",
    "away_team_goal",
]

raw_player_id_columns = [
    *[f"home_player_{number}" for number in range(1, 12)],
    *[f"away_player_{number}" for number in range(1, 12)],
]

formation_coordinate_columns = [
    column
    for column in match_data.columns
    if column.startswith(
        ("home_player_X", "away_player_X", "home_player_Y", "away_player_Y")
    )
]

attribute_date_columns = [
    "home_team_attr_date",
    "away_team_attr_date",
]

target_column = "target"


In [4]:
excluded_predictor_columns = (
    identifier_columns
    + metadata_columns
    + post_match_leakage_columns
    + raw_player_id_columns
    + formation_coordinate_columns
    + attribute_date_columns
)

# Fail clearly if the engineered schema no longer contains an expected field.
required_columns = excluded_predictor_columns + [target_column]
missing_required_columns = sorted(set(required_columns) - set(match_data.columns))

if missing_required_columns:
    raise KeyError(f"Required columns are missing: {missing_required_columns}")

if len(excluded_predictor_columns) != len(set(excluded_predictor_columns)):
    raise ValueError("A column appears in more than one exclusion group.")

exclusion_summary = pd.DataFrame(
    {
        "Reason for exclusion": [
            "Identifiers",
            "Metadata",
            "Post-match leakage variables",
            "Raw player IDs",
            "Raw formation coordinates",
            "Attribute dates used for historical joins",
        ],
        "Columns": [
            len(identifier_columns),
            len(metadata_columns),
            len(post_match_leakage_columns),
            len(raw_player_id_columns),
            len(formation_coordinate_columns),
            len(attribute_date_columns),
        ],
    }
)

display(exclusion_summary)


,Reason for exclusion,Columns
0,Identifiers,6
1,Metadata,4
2,Post-match leakage variables,2
3,Raw player IDs,22
4,Raw formation coordinates,44
5,Attribute dates used for historical joins,2


In [5]:
candidate_features = match_data.drop(
    columns=excluded_predictor_columns + [target_column]
).copy()
target = match_data[target_column].copy()

print("Candidate predictor shape:", candidate_features.shape)
print("Target shape:", target.shape)
print("Target labels:", sorted(target.unique()))

assert candidate_features.shape[1] == 75
assert set(target.unique()) == {"Home Win", "Draw", "Away Win"}


Candidate predictor shape: (17609, 75)
Target shape: (17609,)
Target labels: ['Away Win', 'Draw', 'Home Win']


### Finding

After removing conceptually invalid inputs and separating the target, there
are **75 legitimate candidate predictors**. This is the starting feature set;
model-based feature selection is intentionally left for Notebook 04.


## 4. Inspect Predictor Types and Data Quality

Numerical and categorical columns need different preprocessing. We also check
missing values, infinite values, duplicate match IDs, and exact duplicate
rows before splitting the data.

These checks identify problems; they do not silently drop matches.


In [6]:
categorical_features = candidate_features.select_dtypes(
    include=["object", "string"]
).columns.tolist()
numerical_features = candidate_features.select_dtypes(
    include=np.number
).columns.tolist()

feature_type_summary = pd.DataFrame(
    {
        "Feature type": ["Numerical", "Categorical", "Total"],
        "Number of features": [
            len(numerical_features),
            len(categorical_features),
            candidate_features.shape[1],
        ],
    }
)

display(feature_type_summary)
print("Categorical predictors:")
print(categorical_features)

assert len(numerical_features) == 65
assert len(categorical_features) == 10


,Feature type,Number of features
0,Numerical,65
1,Categorical,10
2,Total,75


Categorical predictors:
['season', 'league_name', 'home_buildUpPlayDribblingClass', 'home_buildUpPlayPositioningClass', 'home_chanceCreationPositioningClass', 'home_defenceDefenderLineClass', 'away_buildUpPlayDribblingClass', 'away_buildUpPlayPositioningClass', 'away_chanceCreationPositioningClass', 'away_defenceDefenderLineClass']


In [7]:
missing_candidate_values = int(candidate_features.isna().sum().sum())
infinite_candidate_values = int(
    np.isinf(candidate_features[numerical_features].to_numpy()).sum()
)
duplicate_match_ids = int(match_data["id"].duplicated().sum())
exact_duplicate_rows = int(match_data.duplicated().sum())

quality_summary = pd.DataFrame(
    {
        "Check": [
            "Missing values in 75 candidate predictors",
            "Infinite values in numerical predictors",
            "Duplicate match IDs",
            "Exact duplicate rows",
        ],
        "Count": [
            missing_candidate_values,
            infinite_candidate_values,
            duplicate_match_ids,
            exact_duplicate_rows,
        ],
    }
)

display(quality_summary)

assert quality_summary["Count"].eq(0).all()


,Check,Count
0,Missing values in 75 candidate predictors,0
1,Infinite values in numerical predictors,0
2,Duplicate match IDs,0
3,Exact duplicate rows,0


### Finding

The candidate predictor matrix contains **65 numerical features and 10
categorical features**. It has no missing or infinite predictor values, and
every match ID is unique. Therefore, no imputation or row deletion is needed.


## 5. Chronological Train, Validation, and Test Split

Football data is time ordered. A random split could train a model on later
seasons and then ask it to predict earlier matches, which would not represent
real forecasting.

We preserve the agreed split:

- **Training:** 2010/2011 to 2013/2014
- **Validation:** 2014/2015
- **Test:** 2015/2016

The validation season supports experimental decisions in Notebook 04. The
test season is reserved for the final evaluation in Notebook 05.


In [8]:
training_seasons = ["2010/2011", "2011/2012", "2012/2013", "2013/2014"]
validation_season = "2014/2015"
test_season = "2015/2016"

expected_seasons = set(training_seasons + [validation_season, test_season])
unexpected_seasons = set(match_data["season"].unique()) - expected_seasons

if unexpected_seasons:
    raise ValueError(f"Unexpected seasons found: {sorted(unexpected_seasons)}")

train_data = match_data[match_data["season"].isin(training_seasons)].copy()
validation_data = match_data[match_data["season"] == validation_season].copy()
test_data = match_data[match_data["season"] == test_season].copy()

assert len(train_data) + len(validation_data) + len(test_data) == len(match_data)


In [9]:
split_summary = pd.DataFrame(
    {
        "Split": ["Training", "Validation", "Test"],
        "Season(s)": [
            "2010/2011–2013/2014",
            validation_season,
            test_season,
        ],
        "Rows": [len(train_data), len(validation_data), len(test_data)],
        "First match": [
            train_data["date"].min().date(),
            validation_data["date"].min().date(),
            test_data["date"].min().date(),
        ],
        "Last match": [
            train_data["date"].max().date(),
            validation_data["date"].max().date(),
            test_data["date"].max().date(),
        ],
    }
)

train_ids = set(train_data["id"])
validation_ids = set(validation_data["id"])
test_ids = set(test_data["id"])

overlap_summary = pd.DataFrame(
    {
        "Pair": [
            "Training and validation",
            "Training and test",
            "Validation and test",
        ],
        "Overlapping match IDs": [
            len(train_ids & validation_ids),
            len(train_ids & test_ids),
            len(validation_ids & test_ids),
        ],
    }
)

display(split_summary)
display(overlap_summary)

assert overlap_summary["Overlapping match IDs"].eq(0).all()
assert split_summary["Rows"].tolist() == [11413, 3067, 3129]


,Split,Season(s),Rows,First match,Last match
0,Training,2010/2011–2013/2014,11413,2010-07-17,2014-05-18
1,Validation,2014/2015,3067,2014-07-18,2015-05-31
2,Test,2015/2016,3129,2015-07-17,2016-05-25


,Pair,Overlapping match IDs
0,Training and validation,0
1,Training and test,0
2,Validation and test,0


In [10]:
feature_columns = candidate_features.columns.tolist()

X_train = train_data[feature_columns].copy()
y_train = train_data[target_column].copy()

X_validation = validation_data[feature_columns].copy()
y_validation = validation_data[target_column].copy()

X_test = test_data[feature_columns].copy()
y_test = test_data[target_column].copy()

# Only training and validation label distributions are inspected here.
class_distribution = pd.concat(
    {
        "Training": y_train.value_counts(normalize=True),
        "Validation": y_validation.value_counts(normalize=True),
    },
    axis=1,
).mul(100).round(2)

class_distribution.index.name = "Match outcome"
display(class_distribution.rename(columns=lambda name: f"{name} (%)"))


,Training (%),Validation (%)
Match outcome,,
Home Win,46.21,44.80
Away Win,28.66,29.54
Draw,25.13,25.66


### Finding

The split contains **11,413 training matches**, **3,067 validation matches**,
and **3,129 test matches**, with no match ID shared between splits. Home wins
are the largest class in training and validation, so Notebook 04 will use
Macro F1 as the primary metric alongside accuracy.

No test labels or test performance are inspected in this notebook.


## 6. Convert Season to a Numeric Time Feature

The text `2010/2011` describes an ordered time period. If season were one-hot
encoded, a future season unseen during training would become an all-zero set
of dummy columns. Instead, we keep the start year (`2010`) as a numeric feature
that preserves the direction of time.


In [11]:
X_train["season_start_year"] = X_train["season"].str[:4].astype(int)
X_validation["season_start_year"] = X_validation["season"].str[:4].astype(int)
X_test["season_start_year"] = X_test["season"].str[:4].astype(int)

X_train = X_train.drop(columns="season")
X_validation = X_validation.drop(columns="season")
X_test = X_test.drop(columns="season")

print("Training season-start years:", sorted(X_train["season_start_year"].unique()))
print(
    "Validation season-start years:",
    sorted(X_validation["season_start_year"].unique()),
)
print("Test season-start years:", sorted(X_test["season_start_year"].unique()))


Training season-start years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013)]
Validation season-start years: [np.int64(2014)]
Test season-start years: [np.int64(2015)]


### Finding

Season is now numeric and ordered. This does not use future results: it only
translates a known calendar label into its start year.


## 7. One-Hot Encode the Remaining Categorical Features

Machine-learning estimators need numeric inputs. One-hot encoding creates a
0/1 column for each retained category.

- `drop="first"` removes one reference category from each field and avoids a
  redundant dummy column.
- `handle_unknown="ignore"` lets a later season contain a category not seen
  during training; it will be represented by zeros for that field's known
  dummy columns.
- The encoder is **fitted only on training data**. Validation and test data
  are transformed using the already learned training categories.


In [12]:
categorical_features_after_split = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

training_cardinality = pd.DataFrame(
    {
        "Categorical feature": categorical_features_after_split,
        "Training categories": [
            X_train[column].nunique(dropna=False)
            for column in categorical_features_after_split
        ],
    }
)

display(training_cardinality)
print("Categorical features to encode:", len(categorical_features_after_split))
assert len(categorical_features_after_split) == 9


,Categorical feature,Training categories
0,league_name,11
1,home_buildUpPlayDribblingClass,1
2,home_buildUpPlayPositioningClass,2
3,home_chanceCreationPositioningClass,2
4,home_defenceDefenderLineClass,2
5,away_buildUpPlayDribblingClass,1
6,away_buildUpPlayPositioningClass,2
7,away_chanceCreationPositioningClass,2
8,away_defenceDefenderLineClass,2


Categorical features to encode: 9


In [13]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
    sparse_output=False,
)

# Learning categories is a training-only operation.
encoder.fit(X_train[categorical_features_after_split])

validation_unknown_categories = []
for column, known_categories in zip(
    categorical_features_after_split,
    encoder.categories_,
):
    validation_values = set(X_validation[column].dropna().unique())
    unseen_values = sorted(validation_values - set(known_categories))
    if unseen_values:
        validation_unknown_categories.append(
            {
                "Feature": column,
                "Validation categories not seen in training": ", ".join(unseen_values),
            }
        )

unknown_category_summary = pd.DataFrame(validation_unknown_categories)
display(unknown_category_summary)

# Unknown categories are expected and deliberately handled by the encoder.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Found unknown categories")
    train_encoded_array = encoder.transform(
        X_train[categorical_features_after_split]
    )
    validation_encoded_array = encoder.transform(
        X_validation[categorical_features_after_split]
    )
    test_encoded_array = encoder.transform(
        X_test[categorical_features_after_split]
    )


,Feature,Validation categories not seen in training
0,home_buildUpPlayDribblingClass,"Lots, Normal"
1,away_buildUpPlayDribblingClass,"Lots, Normal"


In [14]:
encoded_columns = encoder.get_feature_names_out(
    categorical_features_after_split
)

train_encoded = pd.DataFrame(
    train_encoded_array,
    columns=encoded_columns,
    index=X_train.index,
)
validation_encoded = pd.DataFrame(
    validation_encoded_array,
    columns=encoded_columns,
    index=X_validation.index,
)
test_encoded = pd.DataFrame(
    test_encoded_array,
    columns=encoded_columns,
    index=X_test.index,
)

X_train_processed = pd.concat(
    [X_train.drop(columns=categorical_features_after_split), train_encoded],
    axis=1,
)
X_validation_processed = pd.concat(
    [
        X_validation.drop(columns=categorical_features_after_split),
        validation_encoded,
    ],
    axis=1,
)
X_test_processed = pd.concat(
    [X_test.drop(columns=categorical_features_after_split), test_encoded],
    axis=1,
)

encoding_summary = pd.DataFrame(
    {
        "Split": ["Training", "Validation", "Test"],
        "Rows": [
            len(X_train_processed),
            len(X_validation_processed),
            len(X_test_processed),
        ],
        "Numeric columns after encoding": [
            X_train_processed.shape[1],
            X_validation_processed.shape[1],
            X_test_processed.shape[1],
        ],
    }
)

display(encoding_summary)

assert X_train_processed.columns.equals(X_validation_processed.columns)
assert X_train_processed.columns.equals(X_test_processed.columns)
assert X_train_processed.shape[1] == 82


,Split,Rows,Numeric columns after encoding
0,Training,11413,82
1,Validation,3067,82
2,Test,3129,82


### Finding

The training-fitted encoder expands the predictors to **82 numeric columns**.
Some team dribbling labels first appear in the validation period; the summary
above makes that visible. They are safely treated as unknown rather than being
learned from future seasons. The same rule is applied automatically to the
reserved test split.


## 8. Final Quality and Redundancy Checks

All checks that can affect modelling decisions use the training split. We
confirm that encoding introduced no missing, infinite, non-numeric, or
constant training features. We then inspect pairs with an absolute training
correlation above 0.95.

High correlation alone does not make a predictor invalid. It is evidence to
review, not a command to delete every related feature.


In [15]:
processed_quality = pd.DataFrame(
    {
        "Split": ["Training", "Validation", "Test"],
        "Missing values": [
            int(X_train_processed.isna().sum().sum()),
            int(X_validation_processed.isna().sum().sum()),
            int(X_test_processed.isna().sum().sum()),
        ],
        "Infinite values": [
            int(np.isinf(X_train_processed.to_numpy(dtype=float)).sum()),
            int(np.isinf(X_validation_processed.to_numpy(dtype=float)).sum()),
            int(np.isinf(X_test_processed.to_numpy(dtype=float)).sum()),
        ],
        "Non-numeric columns": [
            len(X_train_processed.select_dtypes(exclude=np.number).columns),
            len(X_validation_processed.select_dtypes(exclude=np.number).columns),
            len(X_test_processed.select_dtypes(exclude=np.number).columns),
        ],
    }
)

constant_training_features = [
    column
    for column in X_train_processed.columns
    if X_train_processed[column].nunique(dropna=False) <= 1
]

display(processed_quality)
print("Constant training features:", constant_training_features)

assert processed_quality.iloc[:, 1:].eq(0).all().all()
assert constant_training_features == []


,Split,Missing values,Infinite values,Non-numeric columns
0,Training,0,0,0
1,Validation,0,0,0
2,Test,0,0,0


Constant training features: []


In [16]:
training_correlations = X_train_processed.corr().abs()
high_correlation_rows = []
processed_columns = training_correlations.columns

for later_index in range(1, len(processed_columns)):
    for earlier_index in range(later_index):
        correlation = training_correlations.iloc[later_index, earlier_index]
        if correlation > 0.95:
            high_correlation_rows.append(
                {
                    "Feature 1": processed_columns[later_index],
                    "Feature 2": processed_columns[earlier_index],
                    "Absolute training correlation": correlation,
                }
            )

high_correlation_pairs = pd.DataFrame(high_correlation_rows).sort_values(
    "Absolute training correlation",
    ascending=False,
    ignore_index=True,
)

display(high_correlation_pairs)


,Feature 1,Feature 2,Absolute training correlation
0,away_players_with_attributes,away_players_listed,0.971810
1,home_players_with_attributes,home_players_listed,0.967818
2,home_avg_potential,home_avg_overall_rating,0.953087
3,away_avg_potential,away_avg_overall_rating,0.952421


### Finding

Four training feature pairs exceed 0.95 absolute correlation.

`home_players_listed` and `away_players_listed` are near-duplicates of the
corresponding counts of players with attributes. We remove the two simpler
"listed" counts and retain the more informative coverage counts.

The strong overall-rating/potential relationships are different: both may
still carry useful football meaning. Their value is therefore left for the
validation-based feature experiments in Notebook 04 rather than decided here.


In [17]:
redundant_player_count_features = [
    "home_players_listed",
    "away_players_listed",
]

columns_before_redundancy_removal = set(X_train_processed.columns)

X_train_processed = X_train_processed.drop(
    columns=redundant_player_count_features
)
X_validation_processed = X_validation_processed.drop(
    columns=redundant_player_count_features
)
X_test_processed = X_test_processed.drop(
    columns=redundant_player_count_features
)

removed_columns = columns_before_redundancy_removal - set(
    X_train_processed.columns
)

print("Removed redundant features:", sorted(removed_columns))
print("Final predictor count:", X_train_processed.shape[1])

assert removed_columns == set(redundant_player_count_features)
assert X_train_processed.shape[1] == 80
assert X_train_processed.columns.equals(X_validation_processed.columns)
assert X_train_processed.columns.equals(X_test_processed.columns)


Removed redundant features: ['away_players_listed', 'home_players_listed']
Final predictor count: 80


### Finding

Exactly two redundant player-count features were removed. The final processed
matrix has **80 numeric predictors**. No other feature was removed because of
correlation; statistical and model-based feature selection belongs in
Notebook 04.


## 9. Save the Processed Splits

We now attach the unchanged target labels and save the three generated files.
The engineered input file is never overwritten.

Notebook 04 should use the training and validation files for experiments.
The test file must remain unused until the final evaluation in Notebook 05.


In [18]:
train_ready = X_train_processed.copy()
train_ready[target_column] = y_train

validation_ready = X_validation_processed.copy()
validation_ready[target_column] = y_validation

test_ready = X_test_processed.copy()
test_ready[target_column] = y_test

train_output_path = processed_data_directory / "train_processed.csv"
validation_output_path = processed_data_directory / "validation_processed.csv"
test_output_path = processed_data_directory / "test_processed.csv"

processed_data_directory.mkdir(parents=True, exist_ok=True)

train_ready.to_csv(train_output_path, index=False)
validation_ready.to_csv(validation_output_path, index=False)
test_ready.to_csv(test_output_path, index=False)

saved_split_summary = pd.DataFrame(
    {
        "File": [
            train_output_path.relative_to(project_root),
            validation_output_path.relative_to(project_root),
            test_output_path.relative_to(project_root),
        ],
        "Rows": [len(train_ready), len(validation_ready), len(test_ready)],
        "Predictors": [
            train_ready.shape[1] - 1,
            validation_ready.shape[1] - 1,
            test_ready.shape[1] - 1,
        ],
        "Target included": [
            target_column in train_ready,
            target_column in validation_ready,
            target_column in test_ready,
        ],
    }
)

display(saved_split_summary)


,File,Rows,Predictors,Target included
0,data/processed/train_processed.csv,11413,80,True
1,data/processed/validation_processed.csv,3067,80,True
2,data/processed/test_processed.csv,3129,80,True


In [19]:
# Final safeguards for the files handed to the modelling notebooks.
assert train_ready.shape == (11413, 81)
assert validation_ready.shape == (3067, 81)
assert test_ready.shape == (3129, 81)

assert train_ready.columns.equals(validation_ready.columns)
assert train_ready.columns.equals(test_ready.columns)
assert train_ready.columns[-1] == target_column

assert train_output_path.resolve() != engineered_data_path.resolve()
assert validation_output_path.resolve() != engineered_data_path.resolve()
assert test_output_path.resolve() != engineered_data_path.resolve()

print("All preprocessing checks passed.")


All preprocessing checks passed.


## 10. Preprocessing Summary and Handoff

The preprocessing decisions are now explicit and reproducible:

- The engineered source contains 17,609 matches and 156 columns.
- Conceptual exclusions leave 75 valid candidate predictors: 65 numerical
  and 10 categorical.
- The split is chronological: four training seasons, one validation season,
  and the untouched 2015/2016 test season.
- Season is represented by its numeric start year.
- One-hot encoding is learned only from training data and produces 82 numeric
  columns before redundancy removal.
- Only `home_players_listed` and `away_players_listed` are removed, leaving
  80 processed predictors.
- Missing, infinite, constant, overlap, and schema checks all pass.

**Next:** Notebook 04 can load `train_processed.csv` and
`validation_processed.csv` to compare feature sets and models. It must not use
`test_processed.csv` for selection, tuning, or preprocessing decisions.
